In [ ]:
import pandas as pd
import numpy as np

# 1. Carga inicial del dataset
df = pd.read_csv('clientes_inteligencia_financiera_dirty.csv')
print(f"Dataset original: {df.shape[0]} filas, {df.shape[1]} columnas.")

# ==============================================================================
# PASO 1: Limpieza de Variables Monetarias y Numéricas
# ==============================================================================

# Función auxiliar para limpiar símbolos de moneda y comas
def clean_currency(x):
    if pd.isna(x):
        return np.nan
    return float(str(x).replace('$', '').replace(',', '').strip())

df['ingreso_mensual_usd'] = df['ingreso_mensual_usd'].apply(clean_currency)
df['valor_portafolio_usd'] = df['valor_portafolio_usd'].apply(clean_currency)

# Corrección de valores imposibles o errores de digitación -> NaN
# Un portafolio negativo no tiene sentido en este contexto (salvo deuda específica no indicada)
df.loc[df['valor_portafolio_usd'] < 0, 'valor_portafolio_usd'] = np.nan

# Patrimonio de 999,999,999 es claramente un valor placeholder o de error
df.loc[df['patrimonio_estimado_usd'] > 100_000_000, 'patrimonio_estimado_usd'] = np.nan

# Rendimiento anual del 8500% es un error de decimal o dato corrupto (límite razonable < 100%)
df.loc[(df['rendimiento_anual_pct'] > 100) | (df['rendimiento_anual_pct'] < -50), 'rendimiento_anual_pct'] = np.nan

# Score crediticio de 9500 es imposible (la escala suele ser 300-850 o máx 900)
df.loc[df['score_crediticio'] > 1000, 'score_crediticio'] = np.nan

# ==============================================================================
# PASO 2: Estandarización de Variables Categóricas
# ==============================================================================

# Segmento de cliente: Title Case y manejo de 'Desconocido'
df['segmento_cliente'] = df['segmento_cliente'].astype(str).str.strip().str.title()
df.loc[df['segmento_cliente'] == 'Desconocido', 'segmento_cliente'] = np.nan

# Perfil de riesgo: Title Case
df['perfil_riesgo'] = df['perfil_riesgo'].astype(str).str.strip().str.title()
df.loc[df['perfil_riesgo'] == 'Nan', 'perfil_riesgo'] = np.nan

# Producto principal: Title Case y limpieza de 'N/A'
df['producto_principal'] = df['producto_principal'].astype(str).str.strip().str.title()
df.loc[df['producto_principal'].isin(['N/a', 'Nan']), 'producto_principal'] = np.nan

# Moneda: Estandarización a códigos ISO de 3 letras
map_moneda = {'US Dollar': 'USD', 'mxn': 'MXN', 'Euro': 'EUR'}
df['moneda_cuenta'] = df['moneda_cuenta'].astype(str).str.strip().replace(map_moneda)
df.loc[df['moneda_cuenta'] == 'Nan', 'moneda_cuenta'] = np.nan

# ==============================================================================
# PASO 3: Manejo de Fechas
# ==============================================================================
# Formato mixto para manejar 'YYYY-MM-DD', 'DD/MM/YYYY', 'Aug 2025', etc.
# 'SIN FECHA' o textos irreconocibles se convierten automáticamente en NaT (Not a Time)
df['fecha_alta_cliente'] = pd.to_datetime(df['fecha_alta_cliente'], format='mixed', dayfirst=True, errors='coerce')

# ==============================================================================
# PASO 4: Deduplicación Inteligente (Entity Resolution)
# ==============================================================================
# Hay IDs de cliente duplicados con ligeras variaciones (ej. mayúsculas o un dato extra).
# Estrategia: Conservar el registro que tenga MÁS información (menos valores NaN).

# Contamos cuántos valores no nulos hay por fila
df['non_null_count'] = df.notna().sum(axis=1)

# Ordenamos por id_cliente y luego por cantidad de datos (descendente)
df = df.sort_values(by=['id_cliente', 'non_null_count'], ascending=[True, False])

# Eliminamos duplicados, manteniendo el primero (el más completo)
df = df.drop_duplicates(subset=['id_cliente'], keep='first')

# ==============================================================================
# PASO 5: Limpieza Final y Guardado
# ==============================================================================

# Eliminamos la columna auxiliar de conteo
df = df.drop(columns=['non_null_count']).reset_index(drop=True)

# Estandarizamos el nombre del cliente a Title Case para una presentación profesional
df['nombre_cliente'] = df['nombre_cliente'].astype(str).str.strip().str.title()

# Guardamos el dataset limpio
df.to_csv('clientes_inteligencia_financiera_limpio.csv', index=False, encoding='latin1')

print(f"Dataset limpio: {df.shape[0]} filas, {df.shape[1]} columnas.")
print("\nResumen de valores faltantes (NaN) finales:")
print(df.isnull().sum())
print("\nMuestra de los datos limpios:")
print(df[['id_cliente', 'nombre_cliente', 'segmento_cliente', 'valor_portafolio_usd', 'rendimiento_anual_pct', 'score_crediticio']].head(10))

📊 Dataset original: 84 filas, 12 columnas.
✅ Dataset limpio: 80 filas, 12 columnas.

🔍 Resumen de valores faltantes (NaN) finales:
id_cliente                  0
nombre_cliente              0
segmento_cliente            1
ingreso_mensual_usd         8
patrimonio_estimado_usd     1
valor_portafolio_usd        1
rendimiento_anual_pct       1
producto_principal          1
perfil_riesgo               6
moneda_cuenta               0
fecha_alta_cliente          4
score_crediticio           10
dtype: int64

🔍 Muestra de los datos limpios:
  id_cliente           nombre_cliente segmento_cliente  valor_portafolio_usd  \
0   CLI-1001       María Pérez Torres        Corporate             347635.68   
1   CLI-1002        Camila Gómez Díaz             Pyme              84330.24   
2   CLI-1003       Elena Rivera Pérez        Preferred            1234316.83   
3   CLI-1004    Lucía Pérez Rodríguez        Corporate             740003.96   
4   CLI-1005       Sofía Pérez Romero             Pyme         